# Spike de exploracion — feature `ingest`

> **Paso 4 del flujo (`notebook_writer`).** Prototipo visual de la feature `ingest` para el **gate humano (paso 5)**. Es un **spike de exploracion**, no la fuente de verdad del producto: al pasar a `app/src/`, la spec y el bucle TDD **reescriben** la logica (no se copia-pega).

## Regla inviolable: Datos en Boveda (C-01)
Este notebook usa **exclusivamente datos sinteticos** ("matriz de mentiras": 3-5 filas falsas tipo `test1@correo.com`, `Usuario Uno`), generados dentro del propio notebook en una **carpeta temporal aislada**. **Ningun dato real de cliente** entra aqui: nunca se leen `clients/<CLIENTE>/data/` (excluidos por `.gitignore` `clients/*/data/`).

## Que prototipa
Cada historia `HU-xx` de `definition.md` queda demostrada end-to-end con al menos una salida visible (tabla / print). Referencias: `feature_contract.md`, `system_design.md` (§10 Modo Incremental, §11 medallion + esquema `manifest.json`).

| HU | Que demuestra |
|---|---|
| HU-01 | Ingerir un archivo individual (`.csv`/`.xlsx`) -> copia a bronze + entrada en manifest |
| HU-02 | Ingerir una carpeta (recorrido plano, sin recursion) |
| HU-03 | Mezclar archivos y carpetas en una invocacion |
| HU-04 | Dedupe por `sha256` (idempotencia) |
| HU-05 | Procesamiento parcial por ruta + reporte; tenant inexistente aborta global |
| HU-06 | Nunca interpreta el contenido (delimitador `,`/`;`/`|` da igual) |
| HU-07 | Datos ingeridos fuera de git/indexador (C-01) |
| HU-08 | Nucleo reutilizable + CLI como fachada delgada |

> **Como leer este notebook:** ejecutar **"Run All"** de arriba a abajo. Es determinista y solo usa la biblioteca estandar de Python 3.13 (sin pandas ni openpyxl).

In [1]:
# Utilidades del spike — solo biblioteca estandar (sin pandas/openpyxl/dependencias externas).
# Traza: — (soporte)
import hashlib
import io
import json
import shutil
import tempfile
import zipfile
import fnmatch
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path


def show_table(rows, headers, title=None):
    """Imprime una tabla ASCII simple (evita dependencias)."""
    if title:
        print(f"== {title} ==")
    data = [headers] + rows
    widths = [max(len(str(r[i])) for r in data) for i in range(len(headers))]
    def line(r):
        return " | ".join(str(v).ljust(w) for v, w in zip(r, widths))
    print(line(headers))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(line(r))
    print()


def tree(root, prefix=""):
    """Arbol de archivos en ASCII."""
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("`-- " if last else "|-- ") + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            tree(p, prefix + ("    " if last else "|   "))


print("Utilidades cargadas (stdlib: hashlib, json, shutil, zipfile, pathlib, fnmatch).")

Utilidades cargadas (stdlib: hashlib, json, shutil, zipfile, pathlib, fnmatch).


## Datos sinteticos (matriz de mentiras)

Todo lo que sigue usa **archivos falsos generados dentro del notebook** (`test1@correo.com`, `Usuario Uno`), en una **carpeta temporal aislada**. Nunca se leen ni referencian los datos reales de `clients/<CLIENTE>/data/` (C-01).

Generamos:
- Un `clients_root` temporal con un tenant sintetico (estructura `data/{bronze,silver,gold}` + `manifest.json` vacio, como lo deja `client_scaffold`).
- Una carpeta de "entrega" del cliente con: `.csv`, `.xlsx`, un `.txt` (fuera de allow-list) y una **subcarpeta** (para probar el recorrido plano).

In [2]:
# Genera el tenant SINTETICO y la "matriz de mentiras" (archivos falsos del cliente).
# Traza: — (setup; NINGUN dato real del cliente, C-01)
WORK = Path(tempfile.mkdtemp(prefix="zeroleak_ingest_spike_"))
CLIENTS_ROOT = WORK / "clients"


def scaffold_tenant(clients_root, client):
    """Emula lo que deja client_scaffold: data/{bronze,silver,gold} + manifest.json vacio."""
    t = clients_root / client
    for layer in ("bronze", "silver", "gold"):
        (t / "data" / layer).mkdir(parents=True, exist_ok=True)
    (t / "input").mkdir(parents=True, exist_ok=True)
    mpath = t / "data" / "manifest.json"
    if not mpath.exists():
        mpath.write_text(json.dumps({"version": 1, "files": []}), encoding="utf-8")
    return t


def make_csv(sep):
    """CSV sintetico: 3 filas falsas. `sep` = delimitador (, ; |) para probar HU-06."""
    header = sep.join(["id_cliente", "nombre", "email", "telefono"])
    filas = [
        sep.join(["1", "Usuario Uno",  "test1@correo.com", "300-000-0001"]),
        sep.join(["2", "Usuario Dos",  "test2@correo.com", "300-000-0002"]),
        sep.join(["3", "Usuario Tres", "test3@correo.com", "300-000-0003"]),
    ]
    return ("\n".join([header] + filas) + "\n").encode("utf-8")


def make_xlsx():
    """.xlsx minimo VALIDO (un xlsx es un .zip OOXML). Contenido sintetico, sin PII real."""
    parts = {
        "[Content_Types].xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
            '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            '<Default Extension="xml" ContentType="application/xml"/>'
            '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>'
            '<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>'
            '</Types>',
        "_rels/.rels":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>'
            '</Relationships>',
        "xl/workbook.xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
            'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">'
            '<sheets><sheet name="Hoja1" sheetId="1" r:id="rId1"/></sheets></workbook>',
        "xl/_rels/workbook.xml.rels":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>'
            '</Relationships>',
        "xl/worksheets/sheet1.xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
            '<sheetData><row r="1"><c r="A1" t="inlineStr"><is><t>id_orden</t></is></c></row></sheetData></worksheet>',
    }
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for name, content in parts.items():
            z.writestr(name, content)
    return buf.getvalue()


# --- Carpeta de "entrega" del cliente (drop-off) con mezcla realista ---
DROP = WORK / "entrega_cliente"
(DROP / "subcarpeta").mkdir(parents=True, exist_ok=True)
(DROP / "clientes_fidelizacion.csv").write_bytes(make_csv(","))
(DROP / "liquidacion_rappi.xlsx").write_bytes(make_xlsx())
(DROP / "notas_internas.txt").write_bytes(b"comentarios varios, se ignora en silencio\n")  # fuera de allow-list
(DROP / "subcarpeta" / "historico.csv").write_bytes(make_csv(";"))                          # sin recursion -> ignorado

print("Tenants sinteticos se crearan bajo:", CLIENTS_ROOT)
print("\nEntrega del cliente (drop-off) generada:")
tree(DROP)

Tenants sinteticos se crearan bajo: C:\Users\USUARIO\AppData\Local\Temp\zeroleak_ingest_spike_rg9654o3\clients

Entrega del cliente (drop-off) generada:
|-- subcarpeta/
|   `-- historico.csv
|-- clientes_fidelizacion.csv
|-- liquidacion_rappi.xlsx
`-- notas_internas.txt


## Prototipo del nucleo (HU-08)

La logica de ingesta vive en una **funcion core reutilizable** `ingest_paths(client, paths, clients_root)`, invocable desde tests o una futura API. La CLI `zlk ingest` es solo una **fachada delgada** (`cli_ingest`) que llama al core y traduce su resultado a texto + `exit_code`.

Este mismo nucleo alimenta la demostracion de **todas** las historias que siguen.

In [3]:
# NUCLEO reutilizable: la logica vive aqui (no en la CLI). La CLI sera una fachada delgada.
# Traza: HU-08
ALLOW = {".csv", ".xlsx"}   # allow-list del MVP


class TenantNotFoundError(Exception):
    """Precondicion GLOBAL incumplida: el tenant no existe -> aborta toda la invocacion."""


@dataclass
class Outcome:
    path: str
    status: str            # "ingested" | "duplicate" | "failed"
    reason: str
    sha256: str = ""
    stored_as: str = ""


@dataclass
class IngestResult:
    client: str
    ingested: list = field(default_factory=list)
    duplicates: list = field(default_factory=list)
    failed: list = field(default_factory=list)

    @property
    def exit_code(self):
        # 0 = exito total; 1 = exito parcial o fallo por ruta (procesamiento parcial, HU-05)
        return 1 if self.failed else 0


def _sha256(data):
    return hashlib.sha256(data).hexdigest()


def _manifest_path(tenant):
    return tenant / "data" / "manifest.json"


def _ingest_one(file, tenant, manifest):
    # --- validacion basica AGNOSTICA de formato (no se parsea el contenido) ---
    if not file.is_file():
        return Outcome(str(file), "failed", "el archivo no existe")
    if file.suffix.lower() not in ALLOW:
        return Outcome(str(file), "failed", f"extension fuera de allow-list ({file.suffix or 'sin ext'})")
    data = file.read_bytes()
    if len(data) == 0:
        return Outcome(str(file), "failed", "archivo vacio (0 bytes)")
    digest = _sha256(data)
    # --- dedupe por CONTENIDO (sha256), no por nombre (idempotencia, HU-04) ---
    if any(e["sha256"] == digest for e in manifest["files"]):
        return Outcome(str(file), "duplicate", "contenido ya ingerido (sha256 coincide)", digest)
    # --- copia inmutable byte a byte a bronze (sin parsear) ---
    dest = tenant / "data" / "bronze" / file.name
    # HALLAZGO: mismo nombre + contenido distinto no debe pisar la evidencia previa (bronze es inmutable).
    if dest.exists() and _sha256(dest.read_bytes()) != digest:
        dest = dest.with_name(f"{dest.stem}__{digest[:8]}{dest.suffix}")
    dest.write_bytes(data)
    manifest["files"].append({
        "file": dest.name,
        "sha256": digest,
        "ingested_at": datetime.now().isoformat(timespec="seconds"),
        "status": "pending",
    })
    return Outcome(str(file), "ingested", "registrado en bronze (status=pending)", digest, dest.name)


def ingest_paths(client, paths, clients_root):
    """Core: por cada ruta (archivo o carpeta plana) ingiere lo valido de la allow-list."""
    tenant = Path(clients_root) / client
    # PRECONDICION GLOBAL (HU-05): si el tenant no existe, se aborta antes de tocar nada.
    if not (tenant / "data" / "bronze").is_dir() or not _manifest_path(tenant).exists():
        raise TenantNotFoundError(f"tenant '{client}' inexistente o sin estructura data/bronze + manifest")
    manifest = json.loads(_manifest_path(tenant).read_text(encoding="utf-8"))
    res = IngestResult(client=client)

    def dispatch(o):
        {"ingested": res.ingested, "duplicate": res.duplicates, "failed": res.failed}[o.status].append(o)

    for raw in paths:
        p = Path(raw)
        if p.is_dir():
            for child in sorted(p.iterdir(), key=lambda x: x.name):   # recorrido PLANO, sin recursion
                if child.is_file() and child.suffix.lower() in ALLOW:
                    dispatch(_ingest_one(child, tenant, manifest))
                # subcarpetas y extensiones no permitidas: IGNORADAS EN SILENCIO
        elif p.is_file():
            dispatch(_ingest_one(p, tenant, manifest))
        else:
            dispatch(Outcome(str(p), "failed", "la ruta no existe"))

    _manifest_path(tenant).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return res


def cli_ingest(client, *paths, clients_root):
    """Fachada CLI DELGADA (HU-08): invoca el core y traduce resultado -> texto + exit code."""
    try:
        res = ingest_paths(client, list(paths), clients_root)
    except TenantNotFoundError as e:
        print(f"FALLO GLOBAL: {e}")
        print("exit code: 2")
        return 2
    for o in res.ingested:
        print(f"  OK    {Path(o.path).name}: {o.reason}  [sha256 {o.sha256[:12]}...]  -> bronze/{o.stored_as}")
    for o in res.duplicates:
        print(f"  SKIP  {Path(o.path).name}: {o.reason}")
    for o in res.failed:
        print(f"  FAIL  {o.path}: {o.reason}")
    total = len(res.ingested) + len(res.duplicates) + len(res.failed)
    print(f"resumen: {len(res.ingested)} ingeridos, {len(res.duplicates)} dedup, {len(res.failed)} fallidos de {total} rutas.")
    print(f"exit code: {res.exit_code}  (0=exito total, 1=exito parcial / fallo por ruta)")
    return res.exit_code


def show_manifest(tenant):
    m = json.loads(_manifest_path(tenant).read_text(encoding="utf-8"))
    rows = [[e["file"], e["sha256"][:16] + "...", e["status"], e["ingested_at"]] for e in m["files"]]
    show_table(rows, ["file", "sha256 (trunc)", "status", "ingested_at"], f"manifest.json  ({len(rows)} entradas)")


def show_bronze(tenant):
    print("bronze/:", sorted(p.name for p in (tenant / "data" / "bronze").iterdir()))


# tenants frescos por historia, para que cada demostracion sea autocontenida
_seq = {"n": 0}
def fresh_tenant():
    _seq["n"] += 1
    name = f"SANDUCHERIA_DEMO_{_seq['n']:02d}"
    return scaffold_tenant(CLIENTS_ROOT, name), name


print("Nucleo listo: ingest_paths(client, paths, clients_root) + fachada cli_ingest(...).")

Nucleo listo: ingest_paths(client, paths, clients_root) + fachada cli_ingest(...).


### HU-01 — Ingerir un archivo individual con un comando

`zlk ingest <CLIENTE> <archivo>` copia el archivo intacto a `data/bronze/` y crea una entrada en `manifest.json` con `sha256` correcto y `status: "pending"`.

In [4]:
# HU-01: ingerir UN archivo individual con un solo comando.
# Traza: HU-01
t1, c1 = fresh_tenant()
cli_ingest(c1, str(DROP / "clientes_fidelizacion.csv"), clients_root=CLIENTS_ROOT)
print()
show_bronze(t1)
show_manifest(t1)

# Verificacion notarial: la copia en bronze es identica byte a byte al original.
orig  = _sha256((DROP / "clientes_fidelizacion.csv").read_bytes())
copia = _sha256((t1 / "data" / "bronze" / "clientes_fidelizacion.csv").read_bytes())
print("sha256(original) == sha256(copia en bronze):", orig == copia)

  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 35d8d3d344db...]  -> bronze/clientes_fidelizacion.csv
resumen: 1 ingeridos, 0 dedup, 0 fallidos de 1 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

bronze/: ['clientes_fidelizacion.csv']
== manifest.json  (1 entradas) ==
file                      | sha256 (trunc)      | status  | ingested_at        
--------------------------+---------------------+---------+--------------------
clientes_fidelizacion.csv | 35d8d3d344dba3a1... | pending | 2026-07-13T10:31:32

sha256(original) == sha256(copia en bronze): True


### HU-02 — Ingerir una carpeta de entrega (recorrido plano)

Al pasar una carpeta se ingieren los `.csv`/`.xlsx` de su **primer nivel** y se ignoran en silencio los demas archivos (p. ej. `.txt`) y las **subcarpetas** (sin recursion, D-17).

In [5]:
# HU-02: ingerir una CARPETA (recorrido plano) — solo .csv/.xlsx del primer nivel.
# Traza: HU-02
t2, c2 = fresh_tenant()
cli_ingest(c2, str(DROP), clients_root=CLIENTS_ROOT)
print()
show_bronze(t2)
show_manifest(t2)
print("Ignorados en silencio (no son errores):")
print("  - notas_internas.txt         (extension fuera de allow-list)")
print("  - subcarpeta/historico.csv   (sin recursion: solo primer nivel)")

  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 35d8d3d344db...]  -> bronze/clientes_fidelizacion.csv
  OK    liquidacion_rappi.xlsx: registrado en bronze (status=pending)  [sha256 cc72cf19a24f...]  -> bronze/liquidacion_rappi.xlsx
resumen: 2 ingeridos, 0 dedup, 0 fallidos de 2 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

bronze/: ['clientes_fidelizacion.csv', 'liquidacion_rappi.xlsx']
== manifest.json  (2 entradas) ==
file                      | sha256 (trunc)      | status  | ingested_at        
--------------------------+---------------------+---------+--------------------
clientes_fidelizacion.csv | 35d8d3d344dba3a1... | pending | 2026-07-13T10:31:32
liquidacion_rappi.xlsx    | cc72cf19a24f7ed0... | pending | 2026-07-13T10:31:32

Ignorados en silencio (no son errores):
  - notas_internas.txt         (extension fuera de allow-list)
  - subcarpeta/historico.csv   (sin recursion: solo primer nivel)


### HU-03 — Mezclar archivos y carpetas en una sola invocacion

`zlk ingest <CLIENTE> <ruta1> <ruta2> ...` procesa cualquier combinacion de archivos sueltos y carpetas pasados como argumentos, cubriendo entregas heterogeneas del cliente en un comando.

In [6]:
# HU-03: mezclar ARCHIVOS y CARPETAS como argumentos de una misma invocacion.
# Traza: HU-03
EXTRA = WORK / "entrega_extra"
EXTRA.mkdir(exist_ok=True)
(EXTRA / "inventario.csv").write_bytes(make_csv("|"))
(EXTRA / "ventas_pos.xlsx").write_bytes(make_xlsx())

t3, c3 = fresh_tenant()
# un archivo suelto + una carpeta, en la misma llamada
cli_ingest(c3, str(DROP / "clientes_fidelizacion.csv"), str(EXTRA), clients_root=CLIENTS_ROOT)
print()
show_bronze(t3)
show_manifest(t3)

  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 35d8d3d344db...]  -> bronze/clientes_fidelizacion.csv
  OK    inventario.csv: registrado en bronze (status=pending)  [sha256 4b6d10ada6be...]  -> bronze/inventario.csv
  OK    ventas_pos.xlsx: registrado en bronze (status=pending)  [sha256 cc72cf19a24f...]  -> bronze/ventas_pos.xlsx
resumen: 3 ingeridos, 0 dedup, 0 fallidos de 3 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

bronze/: ['clientes_fidelizacion.csv', 'inventario.csv', 'ventas_pos.xlsx']
== manifest.json  (3 entradas) ==
file                      | sha256 (trunc)      | status  | ingested_at        
--------------------------+---------------------+---------+--------------------
clientes_fidelizacion.csv | 35d8d3d344dba3a1... | pending | 2026-07-13T10:31:32
inventario.csv            | 4b6d10ada6be3591... | pending | 2026-07-13T10:31:32
ventas_pos.xlsx           | cc72cf19a24f7ed0... | pending | 2026-07-13T10:31:32



### HU-04 — Dedupe por `sha256` (idempotencia por contenido)

Reenviar un archivo con **contenido identico** a uno ya registrado no crea copia ni entrada nueva (no-op idempotente): re-ejecutar `ingest` es seguro. En cambio, un archivo con el **mismo nombre pero contenido distinto** si se registra como nueva entrada.

In [7]:
# HU-04: dedupe por sha256 (idempotencia) + mismo nombre / contenido distinto = nueva entrada.
# Traza: HU-04
t4, c4 = fresh_tenant()
src = DROP / "clientes_fidelizacion.csv"

print(">>> 1a ingesta del archivo")
cli_ingest(c4, str(src), clients_root=CLIENTS_ROOT)

print("\n>>> Reenvio del MISMO contenido (re-ejecucion segura, Modo Incremental)")
cli_ingest(c4, str(src), clients_root=CLIENTS_ROOT)

# Mismo nombre de archivo, pero CONTENIDO distinto (una fila extra) -> debe registrarse.
variante = WORK / "clientes_fidelizacion.csv"   # mismo nombre, distinto contenido
variante.write_bytes(make_csv(",") + b"4,Usuario Cuatro,test4@correo.com,300-000-0004\n")
print("\n>>> Mismo nombre, CONTENIDO distinto")
cli_ingest(c4, str(variante), clients_root=CLIENTS_ROOT)

print()
show_bronze(t4)
show_manifest(t4)

>>> 1a ingesta del archivo
  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 35d8d3d344db...]  -> bronze/clientes_fidelizacion.csv
resumen: 1 ingeridos, 0 dedup, 0 fallidos de 1 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

>>> Reenvio del MISMO contenido (re-ejecucion segura, Modo Incremental)
  SKIP  clientes_fidelizacion.csv: contenido ya ingerido (sha256 coincide)
resumen: 0 ingeridos, 1 dedup, 0 fallidos de 1 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

>>> Mismo nombre, CONTENIDO distinto
  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 7a00f3e79f68...]  -> bronze/clientes_fidelizacion__7a00f3e7.csv
resumen: 1 ingeridos, 0 dedup, 0 fallidos de 1 rutas.
exit code: 0  (0=exito total, 1=exito parcial / fallo por ruta)

bronze/: ['clientes_fidelizacion.csv', 'clientes_fidelizacion__7a00f3e7.csv']
== manifest.json  (2 entradas) ==
file                                | 

### HU-05 — Errores claros con procesamiento parcial por ruta

Decision zanjada por el humano: cada ruta se valida de forma independiente. Las validas se ingieren, las invalidas (no existe / vacia / extension no permitida) se omiten **sin corromper** `manifest.json`, y al final se reporta que salio bien y que fallo, con su motivo. **Excepcion**: si el **tenant no existe** (precondicion global), la invocacion aborta por completo antes de procesar rutas.

In [8]:
# HU-05: procesamiento PARCIAL por ruta + reporte de exitos/fallos; tenant inexistente = fallo global.
# Traza: HU-05
t5, c5 = fresh_tenant()
bueno   = DROP / "clientes_fidelizacion.csv"
vacio   = WORK / "vacio.csv";   vacio.write_bytes(b"")                 # 0 bytes -> falla
mal_ext = WORK / "reporte.txt"; mal_ext.write_bytes(b"texto cualquiera")  # ext no permitida -> falla
ausente = WORK / "no_existe.csv"                                       # nunca se crea -> falla

print(">>> Invocacion multi-argumento con rutas validas e invalidas mezcladas:")
code = cli_ingest(c5, str(bueno), str(vacio), str(mal_ext), str(ausente), clients_root=CLIENTS_ROOT)
print()
show_manifest(t5)   # SOLO el archivo bueno quedo registrado; los fallidos no corrompen el ledger

print(">>> Precondicion GLOBAL: tenant inexistente aborta TODA la invocacion (nada se procesa):")
cli_ingest("TENANT_INEXISTENTE", str(bueno), clients_root=CLIENTS_ROOT)

>>> Invocacion multi-argumento con rutas validas e invalidas mezcladas:
  OK    clientes_fidelizacion.csv: registrado en bronze (status=pending)  [sha256 35d8d3d344db...]  -> bronze/clientes_fidelizacion.csv
  FAIL  C:\Users\USUARIO\AppData\Local\Temp\zeroleak_ingest_spike_rg9654o3\vacio.csv: archivo vacio (0 bytes)
  FAIL  C:\Users\USUARIO\AppData\Local\Temp\zeroleak_ingest_spike_rg9654o3\reporte.txt: extension fuera de allow-list (.txt)
  FAIL  C:\Users\USUARIO\AppData\Local\Temp\zeroleak_ingest_spike_rg9654o3\no_existe.csv: la ruta no existe
resumen: 1 ingeridos, 0 dedup, 3 fallidos de 4 rutas.
exit code: 1  (0=exito total, 1=exito parcial / fallo por ruta)

== manifest.json  (1 entradas) ==
file                      | sha256 (trunc)      | status  | ingested_at        
--------------------------+---------------------+---------+--------------------
clientes_fidelizacion.csv | 35d8d3d344dba3a1... | pending | 2026-07-13T10:31:32

>>> Precondicion GLOBAL: tenant inexistente aborta TODA

2

### HU-06 — Archivador notarial: nunca interpreta el contenido

Un `.csv` con delimitador `,`, `;` o `|` se ingiere igual: se copian los bytes y se calcula `sha256`, sin leer ni interpretar la estructura interna. El parseo es problema de una etapa posterior (`load_data`/`validate`).

In [9]:
# HU-06: ingest NO interpreta el contenido — delimitador , ; | da igual (copia bytes + hashea).
# Traza: HU-06
t6, c6 = fresh_tenant()
filas = []
for sep, etiqueta in [(",", "coma"), (";", "punto_y_coma"), ("|", "pipe")]:
    f = WORK / f"delim_{etiqueta}.csv"
    f.write_bytes(make_csv(sep))
    ingest_paths(c6, [str(f)], CLIENTS_ROOT)
    filas.append([etiqueta, f.suffix, _sha256(f.read_bytes())[:12] + "..."])

show_table(filas, ["delimitador", "ext", "sha256(bytes) trunc"],
           "3 CSV con las MISMAS filas logicas pero distinto delimitador")
show_manifest(t6)
print("Los tres se ingirieron sin error: ingest copio bytes y hasheo, sin parsear la estructura.")
print("Nota: copia byte a byte => distinto delimitador produce distinto sha256 (identidad por contenido).")

== 3 CSV con las MISMAS filas logicas pero distinto delimitador ==
delimitador  | ext  | sha256(bytes) trunc
-------------+------+--------------------
coma         | .csv | 35d8d3d344db...    
punto_y_coma | .csv | 8d68c50ded4d...    
pipe         | .csv | 4b6d10ada6be...    

== manifest.json  (3 entradas) ==
file                   | sha256 (trunc)      | status  | ingested_at        
-----------------------+---------------------+---------+--------------------
delim_coma.csv         | 35d8d3d344dba3a1... | pending | 2026-07-13T10:31:32
delim_punto_y_coma.csv | 8d68c50ded4dc99a... | pending | 2026-07-13T10:31:32
delim_pipe.csv         | 4b6d10ada6be3591... | pending | 2026-07-13T10:31:32

Los tres se ingirieron sin error: ingest copio bytes y hasheo, sin parsear la estructura.
Nota: copia byte a byte => distinto delimitador produce distinto sha256 (identidad por contenido).


### HU-07 — Datos ingeridos fuera del control de versiones (C-01)

Tras ingerir, `bronze/` y `manifest.json` viven bajo `clients/<CLIENTE>/data/`, cubierto por la regla `.gitignore` `clients/*/data/`. La Boveda se cumple **por diseno**, no por configuracion opcional.

In [10]:
# HU-07: los datos ingeridos caen bajo la regla .gitignore 'clients/*/data/' (C-01, por diseno).
# Traza: HU-07
GITIGNORE_RULE = "clients/*/data/"   # regla real del repo (.gitignore, C-01)

def cae_en_boveda(rel_path):
    # emula el match de gitignore para la regla de directorio 'clients/*/data/'
    return fnmatch.fnmatch(rel_path, "clients/*/data/*")

rutas = [
    "clients/SANDUCHERIA_DEMO_01/data/bronze/clientes_fidelizacion.csv",  # ingerido (con PII)
    "clients/SANDUCHERIA_DEMO_01/data/manifest.json",                     # ledger
    "clients/SANDUCHERIA_DEMO_01/input/finance.yaml",                     # config: SI se versiona
    "clients/SANDUCHERIA_DEMO_01/client.yaml",                            # identidad: SI se versiona
]
filas = [[r, "IGNORADO (boveda)" if cae_en_boveda(r) else "versionable"] for r in rutas]
show_table(filas, ["ruta relativa en el repo", f"regla '{GITIGNORE_RULE}'"], "Frontera C-01 (Datos en Boveda)")
print("bronze/ y manifest.json quedan fuera de git y del indexador de la IA por diseno.")
print("Este spike corre en carpeta temporal aislada (jamas toca clients/ real):", WORK)

== Frontera C-01 (Datos en Boveda) ==
ruta relativa en el repo                                          | regla 'clients/*/data/'
------------------------------------------------------------------+------------------------
clients/SANDUCHERIA_DEMO_01/data/bronze/clientes_fidelizacion.csv | IGNORADO (boveda)      
clients/SANDUCHERIA_DEMO_01/data/manifest.json                    | IGNORADO (boveda)      
clients/SANDUCHERIA_DEMO_01/input/finance.yaml                    | versionable            
clients/SANDUCHERIA_DEMO_01/client.yaml                           | versionable            

bronze/ y manifest.json quedan fuera de git y del indexador de la IA por diseno.
Este spike corre en carpeta temporal aislada (jamas toca clients/ real): C:\Users\USUARIO\AppData\Local\Temp\zeroleak_ingest_spike_rg9654o3


## Hallazgos del spike (insumo para `spec.md`, paso 6)

**Cobertura de historias** (cada `HU-xx` quedo demostrada con salida visible):

| HU | Demostrado | Evidencia visible |
|---|---|---|
| HU-01 | Ingesta de un archivo con un comando | copia en `bronze/`, entrada en manifest con `status=pending`, igualdad `sha256(original)==sha256(copia)` |
| HU-02 | Ingesta de carpeta (recorrido plano) | solo `.csv`/`.xlsx` del primer nivel; `.txt` y subcarpeta ignorados en silencio |
| HU-03 | Mezcla de archivos y carpetas en una invocacion | los 3 archivos (suelto + carpeta) ingeridos |
| HU-04 | Dedupe por `sha256` + colision de nombre | reenvio identico = no-op idempotente; mismo nombre / distinto contenido = nueva entrada |
| HU-05 | Procesamiento parcial + reporte + fallo global | reporte OK/FAIL por ruta, manifest solo con lo valido, tenant inexistente aborta todo (exit 2) |
| HU-06 | Sin parseo del contenido | 3 CSV con delimitador `,`/`;`/`|` ingeridos sin error, byte a byte |
| HU-07 | Frontera C-01 por diseno | `data/` cae bajo la regla `.gitignore` `clients/*/data/` |
| HU-08 | Nucleo reutilizable + CLI fachada | `ingest_paths(client, paths, clients_root)` con `cli_ingest(...)` delgada encima |

**Enfoques que funcionaron**
- Identidad por **contenido** (`sha256`) y no por nombre: habilita dedupe idempotente (Modo Incremental) de forma natural.
- Nucleo agnostico de formato: leer bytes + hashear + copiar, sin abrir el archivo como CSV/Excel. El `.xlsx` se trato igual que el `.csv`.
- `IngestResult` con listas `ingested / duplicates / failed` traduce limpio a un reporte de exitos/fallos por ruta y a un `exit_code`.

**Riesgos / decisiones que la spec (`CA-xx`) debe precisar**
1. **Colision de nombre en `bronze/`** (hallazgo del spike): dos archivos con **mismo nombre** y **contenido distinto** no pueden compartir ruta en `bronze/` sin pisar la evidencia (viola "copia inmutable, nunca se altera"). El prototipo lo resolvio agregando sufijo `__<sha8>` al nombre almacenado y guardando ese nombre en el campo `file` del manifest. La spec debe **fijar la politica oficial** de nombrado en bronze.
2. **Codigo de salida**: el prototipo usa `0` (exito total), `1` (exito parcial / fallo por ruta), `2` (fallo global por tenant inexistente). La spec debe ratificar la semantica.
3. **Estado de un `duplicate`**: aqui es un no-op idempotente reportado como `SKIP` (no error, no cuenta como fallo). Confirmar en `CA-xx`.
4. **Campos del manifest hoy**: `file`, `sha256`, `ingested_at`, `status:"pending"` (per §11). `period`, `run_id`, `output` quedan fuera hasta una etapa posterior.
5. **Emparejamiento archivo->contrato (D-18)**: fuera de alcance, ninguna celda lo toca (correcto).

**Recordatorio de gate**: este notebook es un **spike de referencia**, no la fuente de verdad. Al pasar a `app/src/` la logica se **reescribe** con la spec y el bucle TDD (no se copia-pega). El **siguiente paso es el gate humano (paso 5)**; no avanzar a `spec_writer` (paso 6) sin aprobacion.
